# Case 4 — Urban Rainfall Monitoring: Extreme Event Detection

## Assignment context

This notebook is part of the Smart Monitoring System assignment for Master students in Civil Engineering and Territorial Protection.

**Monitoring objective:** Flash flood monitoring and civil protection alerting  
**Main issue:** Extreme rainfall event

Tasks:

1. inspect the raw sensor data;
2. detect the issue visually;
3. detect the issue statistically or with ML;
4. decide whether to correct, flag or preserve the observations;
5. prepare the dataset for ingestion, analysis, dashboarding and alerting in istSOS4Things.


## Case description

A rainfall monitoring station is used for urban drainage and flash flood early warning. The dataset contains a simulated extreme rainfall event.

Unlike faulty outliers, this event should not be removed. Students should detect it, characterize it and design an alerting rule.


## 1. Import libraries and load the dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = "dataset_4_extreme_rainfall_event.csv"
VALUE_COL = "rainfall_mm_h"

df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
df.head()


## 2. First inspection


In [ ]:
print(df.info())
print("\nMissing values:")
print(df.isna().sum())
print("\nSummary statistics:")
display(df.describe())


## 3. Visual inspection of raw data


In [ ]:
plt.figure(figsize=(12,4))
plt.plot(df["timestamp"], df[VALUE_COL], marker=".", linewidth=1)
plt.title(f"Raw time series: {VALUE_COL}")
plt.xlabel("Time")
plt.ylabel(VALUE_COL)
plt.grid(True)
plt.show()


## Detection strategy

Suggested checks:

- plot rainfall intensity;
- compute rolling rainfall accumulation;
- compare with operational thresholds;
- detect exceedance duration;
- create alert levels.

Treatment:

- do not correct the event;
- flag it as hazardous;
- compute event statistics;
- use it for alerting and forecasting.


## 4. Rolling rainfall accumulation


In [ ]:
df["rainfall_3h_sum"] = df[VALUE_COL].rolling(window=3).sum()
df["rainfall_6h_sum"] = df[VALUE_COL].rolling(window=6).sum()

plt.figure(figsize=(12,4))
plt.plot(df["timestamp"], df[VALUE_COL], label="hourly rainfall")
plt.plot(df["timestamp"], df["rainfall_3h_sum"], label="3h rolling sum")
plt.plot(df["timestamp"], df["rainfall_6h_sum"], label="6h rolling sum")
plt.title("Rainfall intensity and rolling accumulation")
plt.xlabel("Time")
plt.ylabel("Rainfall [mm]")
plt.legend()
plt.grid(True)
plt.show()


## 5. Threshold-based alert detection


In [ ]:
yellow_threshold = 20
orange_threshold = 40
red_threshold = 60

conditions = [
    df[VALUE_COL] >= red_threshold,
    df[VALUE_COL] >= orange_threshold,
    df[VALUE_COL] >= yellow_threshold
]
choices = ["red", "orange", "yellow"]
df["alert_level"] = np.select(conditions, choices, default="none")

display(df["alert_level"].value_counts())

plt.figure(figsize=(12,4))
plt.plot(df["timestamp"], df[VALUE_COL], label="rainfall")
plt.axhline(yellow_threshold, linestyle="--", label="yellow threshold")
plt.axhline(orange_threshold, linestyle="--", label="orange threshold")
plt.axhline(red_threshold, linestyle="--", label="red threshold")
plt.title("Rainfall alert thresholds")
plt.xlabel("Time")
plt.ylabel("Rainfall [mm/h]")
plt.legend()
plt.grid(True)
plt.show()


## 6. Statistical detection using percentiles


In [ ]:
p95 = df[VALUE_COL].quantile(0.95)
p99 = df[VALUE_COL].quantile(0.99)

df["above_p95"] = df[VALUE_COL] > p95
df["above_p99"] = df[VALUE_COL] > p99

print("95th percentile:", p95)
print("99th percentile:", p99)
print("Observations above p99:", df["above_p99"].sum())

plt.figure(figsize=(12,4))
plt.plot(df["timestamp"], df[VALUE_COL], label="rainfall")
plt.scatter(df.loc[df["above_p99"], "timestamp"], df.loc[df["above_p99"], VALUE_COL], label="above p99")
plt.title("Extreme rainfall detection using percentile threshold")
plt.xlabel("Time")
plt.ylabel("Rainfall [mm/h]")
plt.legend()
plt.grid(True)
plt.show()


## 7. Event characterization and export


In [ ]:
event = df[df["alert_level"] != "none"]

event_summary = {
    "event_start": event["timestamp"].min(),
    "event_end": event["timestamp"].max(),
    "duration_hours": len(event),
    "max_intensity_mm_h": event[VALUE_COL].max(),
    "total_event_rainfall_mm": event[VALUE_COL].sum()
}

display(event_summary)

df["quality_flag"] = np.where(df["alert_level"] != "none", "extreme_event_preserved", "raw")

enriched = df[["timestamp", "rainfall_mm_h", "rainfall_3h_sum", "rainfall_6h_sum", "alert_level", "quality_flag"]]
enriched.to_csv("enriched_dataset_4_extreme_rainfall_event.csv", index=False)
enriched.head()


## Final questions

1. What is the main data quality issue or hazardous event?
2. Which visual method was most useful?
3. Which statistical or ML method was most useful?
4. Which observations should be corrected, removed, flagged or preserved?
5. What would be a suitable alerting rule for an operational dashboard?
6. How would you model this dataset in SensorThings API?
   - Thing
   - Location
   - Sensor
   - ObservedProperty
   - Datastream
   - Observation
